# MiniViGPT — Core Model Smoke Test

Notebook này **không train dataset**. Nó kiểm tra architecture invariant trước khi bạn tốn GPU: parameter count, forward shape, causal prefix invariance và generation shape. Đọc `docs/course/DAY01.md` → `DAY04.md` song song.


In [ ]:
import sys
from pathlib import Path
repo = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(repo / 'src'))

import torch
from minivigpt import MiniViGPT, MiniViGPTConfig
print('PyTorch:', torch.__version__)


## 1. Tạo một model nhỏ

Dùng manual attention để mọi phép toán đúng với phần lý thuyết Day 2. `hidden_dim=None` kích hoạt LLaMA-style SwiGLU width rule.


In [ ]:
cfg = MiniViGPTConfig(
    vocab_size=1000,
    max_seq_len=64,
    dim=128,
    n_layers=2,
    n_heads=4,
    hidden_dim=None,
    ffn_multiple_of=64,
    attention_impl='manual',
)
model = MiniViGPT(cfg).eval()
print(cfg)
print(f'Parameters: {model.num_parameters():,}')


## 2. Forward shape

Invariant: `[B,T] -> [B,T,V]`. Target phải cùng `[B,T]`.


In [ ]:
x = torch.randint(0, cfg.vocab_size, (2, 32))
y = torch.randint(0, cfg.vocab_size, (2, 32))
logits, loss = model(x, y)
print('input :', x.shape)
print('logits:', logits.shape)
print('loss  :', float(loss))
assert logits.shape == (2, 32, cfg.vocab_size)


## 3. Behavioral causal test

Đổi future tokens nhưng giữ prefix. Logits ở prefix phải không đổi.


In [ ]:
torch.manual_seed(7)
a = torch.randint(0, cfg.vocab_size, (1, 16))
b = a.clone()
b[:, 10:] = torch.randint(0, cfg.vocab_size, (1, 6))
la, _ = model(a)
lb, _ = model(b)
max_diff = (la[:, :10] - lb[:, :10]).abs().max().item()
print('max prefix logit diff:', max_diff)
assert max_diff < 1e-5


## 4. Generation shape

Model chưa pretrain nên text/token sinh ra chưa có nghĩa. Cell này chỉ kiểm autoregressive loop tăng sequence đúng số token.


In [ ]:
prompt = x[:1, :5]
out = model.generate(prompt, max_new_tokens=10, temperature=0.0)
print('prompt   :', prompt.shape)
print('generated:', out.shape)
assert out.shape == (1, 15)


## Tiếp theo

- Day 5: dataset/tokenizer.
- Day 6: training loop.
- Day 7: Kaggle CLI debug job.

Đừng chạy config 20M trước khi `pytest -q` và Kaggle debug pass.
